# Cat and Dog image classification

## Import dependencies

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

## Select the device

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else "mps" if torch.backends.mps.is_available() 
    else "cpu"
)
print(f"Active Device: {device}")

print(f"PyTorch version: {torch.__version__}")

if torch.cuda.is_available():
    print("CUDA available")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("CUDA is not available on this machine")

Active Device: cuda
PyTorch version: 2.14.0+cu130
CUDA available
CUDA version: 13.0
GPU name: NVIDIA GeForce RTX 3090
Number of GPUs: 1


## Extract the data

In [3]:
from pathlib import Path
import zipfile
from tqdm import tqdm

zip_path = Path("data/PetImages.zip")
extract_dir = Path("data")

if not (extract_dir / "PetImages").exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        members = zip_ref.infolist()
        for member in tqdm(members, desc="Extracting dataset"):
            zip_ref.extract(member, extract_dir)
    print(f"Dataset extracted to: {extract_dir / 'PetImages'}")
else:
    print(f"Dataset already exists at: {extract_dir / 'PetImages'}")

Extracting dataset:   0%|          | 68/50004 [00:00<03:55, 211.77it/s]

Extracting dataset: 100%|██████████| 50004/50004 [11:22<00:00, 73.28it/s] 

Dataset extracted to: data/PetImages


In [4]:
image_root = extract_dir / "PetImages"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(root=image_root, transform=transform)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=8,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=8,
    pin_memory=True
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Classes: {full_dataset.classes}")

Training samples: 19998
Validation samples: 5000
Classes: ['Cat', 'Dog']


## Setting up the model

In [5]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = SimpleCNN(num_classes=len(full_dataset.classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 5

## Training phase

In [6]:
from tqdm import tqdm

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss / train_total
    train_acc = train_correct / train_total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

print("Training finished.")

Epoch 1/5:  12%|█▏        | 76/625 [00:05<00:13, 39.38it/s]/workspace/catdog_classification/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 1/5: 100%|██████████| 625/625 [00:19<00:00, 32.06it/s]


Epoch 1/5 | Train Loss: 0.6236 | Train Acc: 0.6496 | Val Loss: 0.5407 | Val Acc: 0.7338


Epoch 2/5:  68%|██████▊   | 425/625 [00:07<00:03, 56.67it/s]/workspace/catdog_classification/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 2/5: 100%|██████████| 625/625 [00:11<00:00, 54.31it/s]


Epoch 2/5 | Train Loss: 0.5152 | Train Acc: 0.7482 | Val Loss: 0.4686 | Val Acc: 0.7792


Epoch 3/5:  40%|████      | 252/625 [00:04<00:06, 56.69it/s]/workspace/catdog_classification/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 3/5: 100%|██████████| 625/625 [00:11<00:00, 52.74it/s]


Epoch 3/5 | Train Loss: 0.4264 | Train Acc: 0.8039 | Val Loss: 0.4480 | Val Acc: 0.7946


Epoch 4/5:  78%|███████▊  | 488/625 [00:09<00:02, 57.94it/s]/workspace/catdog_classification/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 4/5: 100%|██████████| 625/625 [00:11<00:00, 52.36it/s]


Epoch 4/5 | Train Loss: 0.3512 | Train Acc: 0.8458 | Val Loss: 0.4426 | Val Acc: 0.8214


Epoch 5/5:   0%|          | 0/625 [00:00<?, ?it/s]/workspace/catdog_classification/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 5/5: 100%|██████████| 625/625 [00:11<00:00, 52.27it/s]


Epoch 5/5 | Train Loss: 0.2786 | Train Acc: 0.8817 | Val Loss: 0.4852 | Val Acc: 0.8122
Training finished.


## Evaluation

In [10]:
model = SimpleCNN(num_classes=2).to(device)
model.load_state_dict(torch.load("./models/cat_dog_cnn.pth", map_location=device))
model.eval()

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=100352, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=2, bias=True)
  )
)